# 05 — Exploratory Data Analysis

This notebook implements the **Data Understanding / Analysis** phase of the CRISP-DM workflow
for the Divvy bike-share project. It picks up where `02_database_design.ipynb`,
`03_data_cleaning_and_loading.ipynb`, and `04_data_quality_assessment.ipynb` leave off, and explores
the star schema loaded into the `divvy` schema (`dim_date`, `dim_station`, `dim_ride_type`,
`dim_member_type`, `fact_trip`, plus the `vw_daily_rides` / `vw_station_flow` analytical views)
inside `divvy_db`.

Rather than pulling the full `fact_trip` table into memory, most of the analysis below is computed
**inside PostgreSQL** with `GROUP BY` / window-function queries and only the aggregated results are
brought into pandas. A few sections (duration distributions, seasonal anomaly checks) pull a bounded
random sample of raw rows where a genuinely row-level view is useful.

### Questions this notebook answers
1. **Member vs. casual rider behavior** — how do subscribers and one-off riders differ?
2. **Ride volume & duration by day, hour, month, and bike type** — when do people ride, and for how long?
3. **Most-used stations & station net flows** — which stations are busiest, and which need rebalancing?
4. **Seasonal patterns & anomalies** — how does ridership move through the year, and where are the outliers?
5. **Round-trip behavior** — how often do riders return to their starting station, and who does it?

### Deliverables
- A set of saved chart images in `eda_outputs/` for reuse in the portfolio write-up
- A compact **EDA summary** (CSV) of the key metrics behind each chart, for quick reference later


### Import Libraries

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import Markdown, display

import config  # same config.py / database.ini reader used in 02, 03 & 04

warnings.filterwarnings("ignore", category=UserWarning)  # psycopg2 + pandas read_sql notice


### Configuration

In [ ]:
# Same base folder convention used in 01/02/03/04 -- update if your local path differs
BASE_FOLDER = r"G:\\My Drive\\divvy_tripdata"
OUTPUT_FOLDER = os.path.join(BASE_FOLDER, "eda_outputs")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

SCHEMA = "divvy"

# Cap on rows pulled for any row-level (non-aggregated) query below, so exploratory
# pulls stay fast even as the warehouse grows past a couple years of trip history.
ROW_SAMPLE_LIMIT = 200_000

# Consistent member-type palette used across every chart in this notebook
MEMBER_COLORS = {"member": "#0b6e4f", "casual": "#e08e00"}
RIDE_TYPE_ORDER = ["classic_bike", "electric_bike", "docked_bike"]

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
MONTH_ORDER = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

sns.set_theme(style="whitegrid")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.titleweight"] = "bold"

# Running list of (metric, value) tuples so every section can contribute to one
# summary table/CSV at the end, instead of duplicating numbers by hand.
summary_rows = []

def log_metric(section, metric, value):
    summary_rows.append({"section": section, "metric": metric, "value": value})


### Database Connection

In [ ]:
def connect_divvy():
    """Connect to the divvy_db database (mirrors 02_database_design.ipynb / 04_data_quality_assessment.ipynb)."""
    conn = None
    try:
        params = config.config_divvy()
        conn = psycopg2.connect(**params)
        conn.autocommit = True
        return conn
    except (Exception, psycopg2.DatabaseError) as error:
        print(error)
        return None


def run_query(conn, sql, params=None):
    """Run a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)


def save_fig(fig, filename):
    """Save a chart into eda_outputs/ at a consistent resolution."""
    path = os.path.join(OUTPUT_FOLDER, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return path


In [ ]:
conn = connect_divvy()
if conn is None:
    raise ConnectionError("Could not connect to divvy_db.")


## Step 1 — Dataset Overview

A quick census of the warehouse before drilling into any single question: total trips, the date
range covered, how many stations and bike types are represented, and the member/casual split.

In [ ]:
overview_sql = f"""
    SELECT
        COUNT(*)                                    AS total_trips,
        MIN(started_at)                             AS first_trip,
        MAX(started_at)                             AS last_trip,
        COUNT(DISTINCT start_station_key)           AS distinct_start_stations,
        COUNT(DISTINCT end_station_key)             AS distinct_end_stations,
        ROUND(AVG(duration_minutes)::numeric, 2)    AS avg_duration_minutes,
        ROUND(SUM(CASE WHEN is_anomalous THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 3) AS pct_flagged_anomalous
    FROM {SCHEMA}.fact_trip
"""
overview = run_query(conn, overview_sql)
display(overview)

for col in overview.columns:
    log_metric("overview", col, overview.iloc[0][col])


In [ ]:
member_mix_sql = f"""
    SELECT mt.member_casual, rt.rideable_type, COUNT(*) AS trips
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_member_type mt ON f.member_type_key = mt.member_type_key
    JOIN {SCHEMA}.dim_ride_type rt   ON f.ride_type_key   = rt.ride_type_key
    GROUP BY mt.member_casual, rt.rideable_type
    ORDER BY mt.member_casual, rt.rideable_type
"""
member_mix = run_query(conn, member_mix_sql)
display(member_mix.pivot(index="rideable_type", columns="member_casual", values="trips").fillna(0).astype(int))


## Step 2 — Member vs. Casual Rider Behavior

Members (annual subscribers) and casual riders (single-ride/day-pass) tend to use the system very
differently — members skew toward short, utilitarian commute trips, while casual riders skew toward
longer, more leisure-oriented trips. This section quantifies that gap.

In [ ]:
rider_summary_sql = f"""
    SELECT
        mt.member_casual,
        COUNT(*)                                              AS trips,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)    AS pct_of_trips,
        ROUND(AVG(f.duration_minutes)::numeric, 2)            AS avg_duration_min,
        ROUND((PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY f.duration_minutes))::numeric, 2) AS median_duration_min,
        ROUND(AVG(CASE WHEN f.is_round_trip THEN 1.0 ELSE 0.0 END) * 100, 2) AS pct_round_trip
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_member_type mt ON f.member_type_key = mt.member_type_key
    GROUP BY mt.member_casual
    ORDER BY trips DESC
"""
rider_summary = run_query(conn, rider_summary_sql)
display(rider_summary)

for _, row in rider_summary.iterrows():
    log_metric("member_vs_casual", f"{row['member_casual']}_trips", row["trips"])
    log_metric("member_vs_casual", f"{row['member_casual']}_avg_duration_min", row["avg_duration_min"])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

colors = [MEMBER_COLORS.get(m, "#888888") for m in rider_summary["member_casual"]]
axes[0].bar(rider_summary["member_casual"], rider_summary["trips"], color=colors)
axes[0].set_title("Total Trips by Rider Type")
axes[0].set_ylabel("Trips")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[1].bar(rider_summary["member_casual"], rider_summary["avg_duration_min"],
            color=colors, alpha=0.85, label="Mean")
axes[1].bar(rider_summary["member_casual"], rider_summary["median_duration_min"],
            color=colors, alpha=0.4, width=0.4, label="Median")
axes[1].set_title("Ride Duration by Rider Type")
axes[1].set_ylabel("Minutes")
axes[1].legend()

plt.tight_layout()
save_fig(fig, "01_member_vs_casual_overview.png")
plt.show()


In [ ]:
bike_pref_sql = f"""
    SELECT mt.member_casual, rt.rideable_type,
           COUNT(*) AS trips,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY mt.member_casual), 1) AS pct_within_rider_type
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_member_type mt ON f.member_type_key = mt.member_type_key
    JOIN {SCHEMA}.dim_ride_type rt   ON f.ride_type_key   = rt.ride_type_key
    GROUP BY mt.member_casual, rt.rideable_type
"""
bike_pref = run_query(conn, bike_pref_sql)
bike_pref_pivot = bike_pref.pivot(index="member_casual", columns="rideable_type", values="pct_within_rider_type").fillna(0)
bike_pref_pivot = bike_pref_pivot[[c for c in RIDE_TYPE_ORDER if c in bike_pref_pivot.columns]]

fig, ax = plt.subplots(figsize=(7, 4.5))
bike_pref_pivot.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
ax.set_title("Bike Type Preference by Rider Type")
ax.set_ylabel("% of trips within rider type")
ax.set_xlabel("")
ax.legend(title="Bike type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
save_fig(fig, "02_bike_type_preference.png")
plt.show()


In [ ]:
# A bounded random sample of raw durations for a distribution view -- percentiles above already
# came straight from SQL, this is purely for the shape of the distribution (skew, long tail).
duration_sample_sql = f"""
    SELECT mt.member_casual, f.duration_minutes
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_member_type mt ON f.member_type_key = mt.member_type_key
    WHERE f.duration_minutes <= 120   -- trim the extreme long tail for readability
    ORDER BY random()
    LIMIT {ROW_SAMPLE_LIMIT}
"""
duration_sample = run_query(conn, duration_sample_sql)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.violinplot(data=duration_sample, x="member_casual", y="duration_minutes",
                palette=MEMBER_COLORS, cut=0, ax=ax)
ax.set_title("Ride Duration Distribution by Rider Type (≤120 min)")
ax.set_xlabel("")
ax.set_ylabel("Duration (minutes)")
plt.tight_layout()
save_fig(fig, "03_duration_distribution.png")
plt.show()


## Step 3 — Ride Volume & Duration by Day, Hour, Month, and Bike Type

`start_hour`, `day_of_week`, and `month_partition` are denormalized directly onto `fact_trip`
(see `02_database_design.ipynb`), so these queries don't need to join `dim_date` at all.

In [ ]:
hourly_sql = f"""
    SELECT f.start_hour, mt.member_casual, COUNT(*) AS trips
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_member_type mt ON f.member_type_key = mt.member_type_key
    GROUP BY f.start_hour, mt.member_casual
    ORDER BY f.start_hour
"""
hourly = run_query(conn, hourly_sql)
hourly_pivot = hourly.pivot(index="start_hour", columns="member_casual", values="trips").fillna(0)

fig, ax = plt.subplots(figsize=(9, 4.5))
for m in hourly_pivot.columns:
    ax.plot(hourly_pivot.index, hourly_pivot[m], marker="o", markersize=3,
            label=m, color=MEMBER_COLORS.get(m, "#888888"))
ax.set_title("Rides by Hour of Day")
ax.set_xlabel("Hour (0–23)")
ax.set_ylabel("Trips")
ax.set_xticks(range(0, 24, 2))
ax.legend(title="Rider type")
plt.tight_layout()
save_fig(fig, "04_rides_by_hour.png")
plt.show()


In [ ]:
dow_sql = f"""
    SELECT f.day_of_week, mt.member_casual, COUNT(*) AS trips
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_member_type mt ON f.member_type_key = mt.member_type_key
    GROUP BY f.day_of_week, mt.member_casual
    ORDER BY f.day_of_week
"""
dow = run_query(conn, dow_sql)
# day_of_week is stored 1=Monday .. 7=Sunday (see dim_date in 02_database_design.ipynb)
dow["day_name"] = dow["day_of_week"].map(dict(enumerate(DAY_ORDER, start=1)))
dow_pivot = dow.pivot(index="day_name", columns="member_casual", values="trips").reindex(DAY_ORDER).fillna(0)

fig, ax = plt.subplots(figsize=(9, 4.5))
dow_pivot.plot(kind="bar", ax=ax, color=[MEMBER_COLORS.get(c, "#888888") for c in dow_pivot.columns])
ax.set_title("Rides by Day of Week")
ax.set_xlabel("")
ax.set_ylabel("Trips")
plt.xticks(rotation=30)
plt.tight_layout()
save_fig(fig, "05_rides_by_day_of_week.png")
plt.show()


In [ ]:
monthly_sql = f"""
    SELECT f.month_partition, mt.member_casual,
           COUNT(*) AS trips,
           ROUND(AVG(f.duration_minutes)::numeric, 2) AS avg_duration_min
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_member_type mt ON f.member_type_key = mt.member_type_key
    GROUP BY f.month_partition, mt.member_casual
    ORDER BY f.month_partition
"""
monthly = run_query(conn, monthly_sql)
monthly["period"] = pd.to_datetime(monthly["month_partition"].astype(str), format="%Y%m")

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

vol_pivot = monthly.pivot(index="period", columns="member_casual", values="trips")
for m in vol_pivot.columns:
    axes[0].plot(vol_pivot.index, vol_pivot[m], marker="o", markersize=3,
                 label=m, color=MEMBER_COLORS.get(m, "#888888"))
axes[0].set_title("Monthly Ride Volume")
axes[0].set_ylabel("Trips")
axes[0].legend(title="Rider type")

dur_pivot = monthly.pivot(index="period", columns="member_casual", values="avg_duration_min")
for m in dur_pivot.columns:
    axes[1].plot(dur_pivot.index, dur_pivot[m], marker="o", markersize=3,
                 label=m, color=MEMBER_COLORS.get(m, "#888888"))
axes[1].set_title("Monthly Average Duration")
axes[1].set_ylabel("Minutes")
axes[1].set_xlabel("Month")

plt.tight_layout()
save_fig(fig, "06_monthly_volume_and_duration.png")
plt.show()


In [ ]:
bike_by_month_sql = f"""
    SELECT f.month_partition, rt.rideable_type, COUNT(*) AS trips
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_ride_type rt ON f.ride_type_key = rt.ride_type_key
    GROUP BY f.month_partition, rt.rideable_type
    ORDER BY f.month_partition
"""
bike_by_month = run_query(conn, bike_by_month_sql)
bike_by_month["period"] = pd.to_datetime(bike_by_month["month_partition"].astype(str), format="%Y%m")
bike_month_pivot = bike_by_month.pivot(index="period", columns="rideable_type", values="trips").fillna(0)
bike_month_pivot = bike_month_pivot[[c for c in RIDE_TYPE_ORDER if c in bike_month_pivot.columns]]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.stackplot(bike_month_pivot.index, bike_month_pivot.T.values, labels=bike_month_pivot.columns,
             colors=sns.color_palette("viridis", n_colors=bike_month_pivot.shape[1]))
ax.set_title("Bike Type Mix Over Time")
ax.set_ylabel("Trips")
ax.legend(title="Bike type", loc="upper left")
plt.tight_layout()
save_fig(fig, "07_bike_type_mix_over_time.png")
plt.show()


## Step 4 — Most-Used Stations & Station Net Flows

`vw_station_flow` (defined in `02_database_design.ipynb`) already computes `rides_started`,
`rides_ended`, and `net_outflow` per station per day; this section rolls that up across the full
history to find the busiest stations and the ones most in need of manual rebalancing (stations that
consistently bleed bikes out, or consistently accumulate them).

In [ ]:
top_start_sql = f"""
    SELECT s.station_name, COUNT(*) AS rides_started
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_station s ON f.start_station_key = s.station_key
    GROUP BY s.station_name
    ORDER BY rides_started DESC
    LIMIT 15
"""
top_start = run_query(conn, top_start_sql)

top_end_sql = f"""
    SELECT s.station_name, COUNT(*) AS rides_ended
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_station s ON f.end_station_key = s.station_key
    GROUP BY s.station_name
    ORDER BY rides_ended DESC
    LIMIT 15
"""
top_end = run_query(conn, top_end_sql)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].barh(top_start["station_name"][::-1], top_start["rides_started"][::-1], color="#0b6e4f")
axes[0].set_title("Top 15 Stations — Rides Started")
axes[0].set_xlabel("Trips")

axes[1].barh(top_end["station_name"][::-1], top_end["rides_ended"][::-1], color="#1f6fb2")
axes[1].set_title("Top 15 Stations — Rides Ended")
axes[1].set_xlabel("Trips")

plt.tight_layout()
save_fig(fig, "08_top_stations.png")
plt.show()


In [ ]:
station_flow_sql = f"""
    SELECT station_name, SUM(rides_started) AS total_started,
           SUM(rides_ended) AS total_ended, SUM(net_outflow) AS total_net_outflow
    FROM {SCHEMA}.vw_station_flow
    GROUP BY station_name
    HAVING SUM(rides_started) + SUM(rides_ended) >= 100   -- ignore very low-traffic stations
    ORDER BY total_net_outflow DESC
"""
station_flow = run_query(conn, station_flow_sql)

exporters = station_flow.head(10)             # bleed bikes out -- net pickup candidates
importers = station_flow.tail(10).iloc[::-1]  # accumulate bikes -- net drop-off candidates

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].barh(exporters["station_name"][::-1], exporters["total_net_outflow"][::-1], color="#c62828")
axes[0].set_title("Top 10 Net Exporters (more starts than ends)")
axes[0].set_xlabel("Net outflow (trips)")

axes[1].barh(importers["station_name"][::-1], importers["total_net_outflow"][::-1], color="#2e7d32")
axes[1].set_title("Top 10 Net Importers (more ends than starts)")
axes[1].set_xlabel("Net outflow (trips, negative = surplus)")

plt.tight_layout()
save_fig(fig, "09_station_net_flow.png")
plt.show()

log_metric("stations", "busiest_start_station", top_start.iloc[0]["station_name"])
log_metric("stations", "top_net_exporter", exporters.iloc[0]["station_name"])
log_metric("stations", "top_net_importer", importers.iloc[0]["station_name"])


In [ ]:
# Geospatial view: station locations sized/colored by how imbalanced their net flow is
station_geo_sql = f"""
    SELECT s.station_name, s.latitude, s.longitude,
           COALESCE(SUM(vf.net_outflow), 0) AS total_net_outflow
    FROM {SCHEMA}.dim_station s
    LEFT JOIN {SCHEMA}.vw_station_flow vf ON vf.station_id = s.station_id
    WHERE s.latitude IS NOT NULL AND s.longitude IS NOT NULL
    GROUP BY s.station_name, s.latitude, s.longitude
"""
station_geo = run_query(conn, station_geo_sql)

fig, ax = plt.subplots(figsize=(7, 7))
sc = ax.scatter(station_geo["longitude"], station_geo["latitude"],
                 c=station_geo["total_net_outflow"], cmap="coolwarm",
                 s=18, alpha=0.75, vmin=-station_geo["total_net_outflow"].abs().quantile(0.98),
                 vmax=station_geo["total_net_outflow"].abs().quantile(0.98))
ax.set_title("Station Locations Colored by Net Outflow\n(red = exports bikes, blue = accumulates bikes)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="datalim")
plt.colorbar(sc, ax=ax, label="Net outflow (trips)")
plt.tight_layout()
save_fig(fig, "10_station_map_net_flow.png")
plt.show()


## Step 5 — Seasonal Patterns & Anomalies

Chicago's climate makes bike-share ridership strongly seasonal (low in winter, peaking in summer).
This section looks at the daily trend with a rolling average to separate the seasonal signal from
day-to-day noise, then checks the `is_anomalous` flag that `03_data_cleaning_and_loading.ipynb`
writes onto `fact_trip` for rows that failed a sanity rule during ETL.

In [ ]:
daily_sql = f"""
    SELECT full_date, SUM(ride_count) AS trips
    FROM {SCHEMA}.vw_daily_rides
    GROUP BY full_date
    ORDER BY full_date
"""
daily = run_query(conn, daily_sql)
daily["full_date"] = pd.to_datetime(daily["full_date"])
daily["rolling_7d"] = daily["trips"].rolling(7, min_periods=1, center=True).mean()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(daily["full_date"], daily["trips"], color="#bbbbbb", linewidth=0.8, label="Daily trips")
ax.plot(daily["full_date"], daily["rolling_7d"], color="#0b6e4f", linewidth=2, label="7-day rolling avg")
ax.set_title("Daily Ride Volume with Seasonal Trend")
ax.set_ylabel("Trips")
ax.legend()
plt.tight_layout()
save_fig(fig, "11_daily_trend_seasonality.png")
plt.show()


In [ ]:
heatmap_sql = f"""
    SELECT f.day_of_week, f.start_hour, COUNT(*) AS trips
    FROM {SCHEMA}.fact_trip f
    GROUP BY f.day_of_week, f.start_hour
"""
heat = run_query(conn, heatmap_sql)
heat["day_name"] = heat["day_of_week"].map(dict(enumerate(DAY_ORDER, start=1)))
heat_pivot = heat.pivot(index="day_name", columns="start_hour", values="trips").reindex(DAY_ORDER)

fig, ax = plt.subplots(figsize=(12, 4.5))
sns.heatmap(heat_pivot, cmap="YlGnBu", ax=ax, cbar_kws={"label": "Trips"})
ax.set_title("Ride Volume Heatmap — Day of Week × Hour of Day")
ax.set_xlabel("Hour of day")
ax.set_ylabel("")
plt.tight_layout()
save_fig(fig, "12_day_hour_heatmap.png")
plt.show()


In [ ]:
anomaly_sql = f"""
    SELECT f.month_partition,
           COUNT(*) AS trips,
           SUM(CASE WHEN f.is_anomalous THEN 1 ELSE 0 END) AS anomalous_trips,
           ROUND(SUM(CASE WHEN f.is_anomalous THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 3) AS pct_anomalous
    FROM {SCHEMA}.fact_trip f
    GROUP BY f.month_partition
    ORDER BY f.month_partition
"""
anomaly_by_month = run_query(conn, anomaly_sql)
anomaly_by_month["period"] = pd.to_datetime(anomaly_by_month["month_partition"].astype(str), format="%Y%m")
display(anomaly_by_month[["period", "trips", "anomalous_trips", "pct_anomalous"]])

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(anomaly_by_month["period"], anomaly_by_month["pct_anomalous"], width=20, color="#c62828")
ax.set_title("Share of Trips Flagged Anomalous by Month")
ax.set_ylabel("% of trips")
plt.tight_layout()
save_fig(fig, "13_anomaly_rate_by_month.png")
plt.show()

log_metric("anomalies", "overall_pct_anomalous", overview.iloc[0]["pct_flagged_anomalous"])
log_metric("anomalies", "worst_month_pct_anomalous", anomaly_by_month["pct_anomalous"].max())


In [ ]:
# Duration outliers by month -- looks for months where the spread widens noticeably,
# which is often the first sign of a bad batch of source data rather than genuine ridership change.
dur_outlier_sql = f"""
    SELECT f.month_partition, mt.member_casual, f.duration_minutes
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_member_type mt ON f.member_type_key = mt.member_type_key
    WHERE f.duration_minutes <= 180
    ORDER BY random()
    LIMIT {ROW_SAMPLE_LIMIT}
"""
dur_outlier = run_query(conn, dur_outlier_sql)
dur_outlier["period"] = pd.to_datetime(dur_outlier["month_partition"].astype(str), format="%Y%m")

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=dur_outlier, x="period", y="duration_minutes", ax=ax, color="#8ecae6", fliersize=1)
ax.set_title("Ride Duration Spread by Month (≤180 min, sampled)")
ax.set_xlabel("")
ax.set_ylabel("Duration (minutes)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
save_fig(fig, "14_duration_spread_by_month.png")
plt.show()


## Step 6 — Round-Trip Behavior

`is_round_trip` (start station == end station) is computed once during ETL and stored directly on
`fact_trip`. Round trips are a useful proxy for leisure/exercise usage as opposed to point-to-point
commuting.

In [ ]:
round_trip_sql = f"""
    SELECT mt.member_casual, rt.rideable_type,
           COUNT(*) AS trips,
           SUM(CASE WHEN f.is_round_trip THEN 1 ELSE 0 END) AS round_trips,
           ROUND(SUM(CASE WHEN f.is_round_trip THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_round_trip,
           ROUND(AVG(CASE WHEN f.is_round_trip THEN f.duration_minutes END)::numeric, 2) AS avg_round_trip_min,
           ROUND(AVG(CASE WHEN NOT f.is_round_trip THEN f.duration_minutes END)::numeric, 2) AS avg_one_way_min
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_member_type mt ON f.member_type_key = mt.member_type_key
    JOIN {SCHEMA}.dim_ride_type rt   ON f.ride_type_key   = rt.ride_type_key
    GROUP BY mt.member_casual, rt.rideable_type
    ORDER BY pct_round_trip DESC
"""
round_trip = run_query(conn, round_trip_sql)
display(round_trip)

log_metric("round_trip", "overall_pct_round_trip", rider_summary["pct_round_trip"].mean().round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

rt_by_member = round_trip.groupby("member_casual")[["trips", "round_trips"]].sum()
rt_by_member["pct_round_trip"] = (rt_by_member["round_trips"] * 100 / rt_by_member["trips"]).round(2)
colors = [MEMBER_COLORS.get(m, "#888888") for m in rt_by_member.index]
axes[0].bar(rt_by_member.index, rt_by_member["pct_round_trip"], color=colors)
axes[0].set_title("% Round Trips by Rider Type")
axes[0].set_ylabel("% of trips")

durations = round_trip.groupby("member_casual")[["avg_round_trip_min", "avg_one_way_min"]].mean()
durations.plot(kind="bar", ax=axes[1], color=["#6a4c93", "#1f6fb2"])
axes[1].set_title("Avg Duration: Round Trip vs. One-Way")
axes[1].set_ylabel("Minutes")
axes[1].set_xlabel("")
plt.xticks(rotation=0)

plt.tight_layout()
save_fig(fig, "15_round_trip_summary.png")
plt.show()


In [ ]:
round_trip_stations_sql = f"""
    SELECT s.station_name, COUNT(*) AS round_trips
    FROM {SCHEMA}.fact_trip f
    JOIN {SCHEMA}.dim_station s ON f.start_station_key = s.station_key
    WHERE f.is_round_trip
    GROUP BY s.station_name
    ORDER BY round_trips DESC
    LIMIT 15
"""
round_trip_stations = run_query(conn, round_trip_stations_sql)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(round_trip_stations["station_name"][::-1], round_trip_stations["round_trips"][::-1], color="#6a4c93")
ax.set_title("Top 15 Stations for Round Trips")
ax.set_xlabel("Round trips")
plt.tight_layout()
save_fig(fig, "16_top_round_trip_stations.png")
plt.show()

log_metric("round_trip", "top_round_trip_station", round_trip_stations.iloc[0]["station_name"])


## Step 7 — EDA Summary

In [ ]:
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUTPUT_FOLDER, "eda_summary_metrics.csv")
summary_df.to_csv(summary_path, index=False)
display(summary_df)
print(f"Summary metrics saved to: {summary_path}")
print(f"Charts saved to: {OUTPUT_FOLDER}")


In [ ]:
conn.close()


## Key Takeaways

*Fill in after running this notebook against the loaded warehouse:*

- **Member vs. casual:** which group rides more often, and which rides longer? What does that imply
  about how each group uses the system (commuting vs. leisure)?
- **Timing:** do members show a clear AM/PM commute double-peak on weekdays that casual riders lack?
  How does the weekday/weekend mix differ between the two groups?
- **Seasonality:** how much does ridership swing between the lowest and highest month? Are there any
  days that break the seasonal pattern (data issue, weather event, holiday)?
- **Stations:** which stations are the clearest rebalancing priorities (consistent net exporters or
  importers), and do they cluster in a particular part of the service area (downtown vs. residential)?
- **Round trips:** which rider type and bike type combination has the highest round-trip share, and
  does that support a leisure-usage hypothesis?
- **Anomalies:** did the `is_anomalous` rate spike in any particular month, and does that line up with
  a known gap or issue from `04_data_quality_assessment.ipynb`?

These findings feed directly into the next stage of the portfolio project (rider segmentation /
modeling), so it's worth capturing the concrete numbers behind each bullet above rather than just the
direction of the pattern.
